# Continued training on repeat measurements

Restore an earlier checkpoint and continue training on repeat measurements. The matching retake2 input data and checkpoint are required.

See [reproduction notes](../docs/REPRODUCIBILITY.md) for inputs, execution order, and experimental assumptions.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "scripts" / "project_paths.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter from the repository root or its notebooks directory.")
sys.path.insert(0, str(ROOT / "scripts"))
from project_paths import create_run
DATA_DIR, OUTPUT_DIR = create_run('transfer_learning')


## Initialize the research environment

Original code cell 1.


In [ ]:
seed_value = 666

import numpy as np
import random
import tensorflow as tf
# Set random seeds for reproducibility
np.random.seed(seed_value)
random.seed(seed_value)
tf.random.set_seed(seed_value)

# For PyTorch, if used (commented if not applicable)
try:
    import torch
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
except ImportError:
    pass

# Fixing thread settings to enforce determinism (optional)
import os
os.environ['TF_DETERMINISTIC_OPS'] = '1'

# Explicitly set initializers for TensorFlow/Keras models
from tensorflow.keras import initializers

initializer = initializers.GlorotUniform(seed=seed_value)


import numpy as np
import random
import tensorflow as tf

# Set random seeds for reproducibility
np.random.seed(seed_value)
random.seed(seed_value)
tf.random.set_seed(seed_value)

# For PyTorch, if used (commented if not applicable)
try:
    import torch
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
except ImportError:
    pass

# For scikit-learn, ensure random_state is set where applicable


import numpy as np
import random
import tensorflow as tf

# Set random seeds for reproducibility
np.random.seed(seed_value)
random.seed(seed_value)
tf.random.set_seed(seed_value)


import numpy as np
import random

# Set random seeds for reproducibility
np.random.seed(seed_value)
random.seed(seed_value)

import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
import pandas as pd
import os
import csv


## Load augmented repeat-measurement training data

Original code cell 2.


In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt


with h5py.File(str(DATA_DIR / 'processed/ml_20condition_retake2_with_aug_data.h5'), "r") as h5f:
    X, X_cols = h5f["X_with_meta"][:], [s.decode('utf-8') for s in h5f["X_columns"][:]]

with h5py.File(str(DATA_DIR / 'processed/ml_20condition_retake2_with_aug_label.h5'), "r") as h5f:
    Y, Y_cols = h5f["Y_with_meta"][:], [s.decode('utf-8') for s in h5f["Y_columns"][:]]

print("X shape:", X.shape)
print("Y shape:", Y.shape)


wavelength = np.array([float(w) for w in X_cols[4:]])


training_condition = X[:, :4]
training_spectra = X[:, 4:] / 60000
training_target = Y[:, 4:] / 60000

print("training_condition shape:", training_condition.shape)
print("training_spectra shape:", training_spectra.shape)
print("training_target shape:", training_target.shape)


def get_index_range_from_array(start_wl, end_wl, wavelength_array):
    idx_start = np.searchsorted(wavelength_array, start_wl, side='left')
    idx_end = np.searchsorted(wavelength_array, end_wl, side='right')
    return idx_start, idx_end


Zn_index = get_index_range_from_array(212.129, 216.777, wavelength)
Ni_index = get_index_range_from_array(229.563, 234.734, wavelength)
Cu_index = get_index_range_from_array(326.669, 330.016, wavelength)

training_zn_indices = np.arange(Zn_index[0], Zn_index[1])
training_ni_indices = np.arange(Ni_index[0], Ni_index[1])
training_cu_indices = np.arange(Cu_index[0], Cu_index[1])

training_metal_indices = np.concatenate([
    training_zn_indices,
    training_ni_indices,
    training_cu_indices
])


training_background_spectra = training_spectra.copy()
training_background_target = training_target.copy()
training_background_spectra[:, training_metal_indices] = 0
training_background_target[:, training_metal_indices] = 0


plt.figure(figsize=(12, 4))
plt.plot(wavelength, training_spectra[0], label='Original Spectrum')
plt.plot(wavelength, training_background_spectra[0], label='Background Spectrum')

plt.title("Training Sample #0: Full Spectrum vs Background", fontsize=18)
plt.xlabel("Wavelength (nm)", fontsize=14)
plt.ylabel("Intensity", fontsize=14)
plt.legend(fontsize=12, loc='upper right')
plt.grid(True)
plt.tight_layout()
plt.show()

# === 8. Zn + Ni + Cu → Reference ===
training_reference = np.concatenate([
    training_spectra[:, training_zn_indices],
    training_spectra[:, training_ni_indices],
    training_spectra[:, training_cu_indices]
], axis=1)

print("training_reference shape:", training_reference.shape)

plt.figure(figsize=(12, 4))
plt.plot(training_reference[0], label='Reference (Zn + Ni + Cu)')

plt.title("Training Sample #0: Zn + Ni + Cu Reference", fontsize=18)
plt.xlabel("Index", fontsize=14)
plt.ylabel("Intensity", fontsize=14)
plt.legend(fontsize=12, loc='upper right')
plt.grid(True)
plt.tight_layout()
plt.show()


## Combine the test datasets

Original code cell 3.


In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt


def load_h5_with_columns(filepath, data_key="X_with_meta", col_key="X_columns"):
    with h5py.File(filepath, "r") as f:
        data = f[data_key][:]
        columns = [s.decode('utf-8') for s in f[col_key][:]]
    return data, columns


X1, X1_cols = load_h5_with_columns(str(DATA_DIR / 'processed/ml_20condition_testing_data1.h5'))
X2, X2_cols = load_h5_with_columns(str(DATA_DIR / 'processed/ml_20condition_testing_data2.h5'))


#    1 ~ 1.5 → 16,   2 ~ 2.5 → 17,   3 ~ 3.5 → 18
cr = X2[:, 0]
X2[(cr >= 1) & (cr <= 1.5), 0] = 16
X2[(cr >= 2) & (cr <= 2.5), 0] = 17
X2[(cr >= 3) & (cr <= 3.5), 0] = 18
# ────────────────────────────────────

X  = np.vstack([X1, X2])
X_cols = X1_cols


Y1, Y1_cols = load_h5_with_columns(
    str(DATA_DIR / 'processed/ml_20condition_testing_label1.h5'),
    data_key="Y_with_meta", col_key="Y_columns")
Y2, Y2_cols = load_h5_with_columns(
    str(DATA_DIR / 'processed/ml_20condition_testing_label2.h5'),
    data_key="Y_with_meta", col_key="Y_columns")


cr_y = Y2[:, 0]
Y2[(cr_y >= 1) & (cr_y <= 1.5), 0] = 16
Y2[(cr_y >= 2) & (cr_y <= 2.5), 0] = 17
Y2[(cr_y >= 3) & (cr_y <= 3.5), 0] = 18
# ────────────────────────────────────

Y  = np.vstack([Y1, Y2])
Y_cols = Y1_cols

print("X shape:", X.shape)
print("Y shape:", Y.shape)


wavelength = np.array([float(w) for w in X_cols[4:]])


testing_condition = X[:, :4]
testing_spectra = X[:, 4:] / 60000
testing_target = Y[:, 4:] / 60000

print("testing_condition shape:", testing_condition.shape)
print("testing_spectra shape:", testing_spectra.shape)
print("testing_target shape:", testing_target.shape)


def get_index_range_from_array(start_wl, end_wl, wavelength_array):
    idx_start = np.searchsorted(wavelength_array, start_wl, side='left')
    idx_end = np.searchsorted(wavelength_array, end_wl, side='right')
    return idx_start, idx_end


Zn_index = get_index_range_from_array(212.129, 216.777, wavelength)
Ni_index = get_index_range_from_array(229.563, 234.734, wavelength)
Cu_index = get_index_range_from_array(326.669, 330.016, wavelength)

testing_zn_indices = np.arange(Zn_index[0], Zn_index[1])
testing_ni_indices = np.arange(Ni_index[0], Ni_index[1])
testing_cu_indices = np.arange(Cu_index[0], Cu_index[1])

testing_metal_indices = np.concatenate([
    testing_zn_indices,
    testing_ni_indices,
    testing_cu_indices
])


testing_background_spectra = testing_spectra.copy()
testing_background_target = testing_target.copy()
testing_background_spectra[:, testing_metal_indices] = 0
testing_background_target[:, testing_metal_indices] = 0


plt.figure(figsize=(12, 4))
plt.plot(wavelength, testing_spectra[0], label='Original Spectrum')
plt.plot(wavelength, testing_background_spectra[0], label='Background Spectrum')

plt.title("Testing Sample #0: Full Spectrum vs Background", fontsize=18)
plt.xlabel("Wavelength (nm)", fontsize=14)
plt.ylabel("Intensity", fontsize=14)
plt.legend(fontsize=12, loc='upper right')
plt.grid(True)
plt.tight_layout()
plt.show()

# === 10. Zn + Ni + Cu → Reference ===
testing_reference = np.concatenate([
    testing_spectra[:, testing_zn_indices],
    testing_spectra[:, testing_ni_indices],
    testing_spectra[:, testing_cu_indices]
], axis=1)

print("testing_reference shape:", testing_reference.shape)

plt.figure(figsize=(12, 4))
plt.plot(testing_reference[0], label='Reference (Zn + Ni + Cu)')

plt.title("Testing Sample #0: Zn + Ni + Cu Reference", fontsize=18)
plt.xlabel("Index", fontsize=14)
plt.ylabel("Intensity", fontsize=14)
plt.legend(fontsize=12, loc='upper right')
plt.grid(True)
plt.tight_layout()
plt.show()


## Load the earlier repeat-measurement dataset

Original code cell 4.


In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt


with h5py.File(str(DATA_DIR / 'processed/ml_20condition_retake2_data.h5'), "r") as h5f:
    X, X_cols = h5f["X_with_meta"][:], [s.decode('utf-8') for s in h5f["X_columns"][:]]

with h5py.File(str(DATA_DIR / 'processed/ml_20condition_retake2_label.h5'), "r") as h5f:
    Y, Y_cols = h5f["Y_with_meta"][:], [s.decode('utf-8') for s in h5f["Y_columns"][:]]

print("X shape:", X.shape)
print("Y shape:", Y.shape)


wavelength = np.array([float(w) for w in X_cols[4:]])


retake_condition = X[:, :4]
retake_spectra = X[:, 4:] / 60000
retake_target = Y[:, 4:] / 60000


## Load wastewater spectra

Original code cell 5.


In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt


with h5py.File(str(DATA_DIR / 'processed/ml_20condition_wastewater2_data.h5'), "r") as h5f:
    X, X_cols = h5f["X_with_meta"][:], [s.decode('utf-8') for s in h5f["X_columns"][:]]

with h5py.File(str(DATA_DIR / 'processed/ml_20condition_wastewater2_label.h5'), "r") as h5f:
    Y, Y_cols = h5f["Y_with_meta"][:], [s.decode('utf-8') for s in h5f["Y_columns"][:]]

print("X shape:", X.shape)
print("Y shape:", Y.shape)


wavelength = np.array([float(w) for w in X_cols[4:]])


wastewater_condition = X[:, :4]
wastewater_spectra = X[:, 4:] / 60000
wastewater_target = Y[:, 4:] / 60000


## Build the conditional GAN

Original code cell 6.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import regularizers


training_spectra = tf.convert_to_tensor(training_spectra, dtype=tf.float32)
training_target = tf.convert_to_tensor(training_target, dtype=tf.float32)


batch_size = 128
learning_rate = 1e-4
num_epochs = 1000


import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy


import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy


def crop_to_match(src, tgt):
    def _crop(tensors):
        s, t = tensors
        return s[:, :tf.shape(t)[1], :]
    return layers.Lambda(_crop)([src, tgt])
# -------------------------------------

# ---------- Generator ----------
def build_generator(ref_dim, out_dim):
    inp = Input(shape=(ref_dim,), name="interfered_spectrum")
    x   = layers.Reshape((ref_dim, 1))(inp)

    # ===== Encoder =====
    e1 = layers.Conv1D(64, 4, 2, padding="same")(x)
    e1 = layers.BatchNormalization()(e1); e1 = layers.LeakyReLU(0.2)(e1)

    e2 = layers.Conv1D(128, 4, 2, padding="same")(e1)
    e2 = layers.BatchNormalization()(e2); e2 = layers.LeakyReLU(0.2)(e2)

    e3 = layers.Conv1D(256, 4, 2, padding="same")(e2)
    e3 = layers.BatchNormalization()(e3); e3 = layers.LeakyReLU(0.2)(e3)

    e4 = layers.Conv1D(512, 4, 2, padding="same")(e3)
    e4 = layers.BatchNormalization()(e4); e4 = layers.LeakyReLU(0.2)(e4)

    # ===== Bottleneck =====
    b = layers.Conv1D(512, 4, 2, padding="same")(e4)
    b = layers.BatchNormalization()(b); b = layers.LeakyReLU(0.2)(b)

    # ===== Decoder =====
    d4 = layers.Conv1DTranspose(512, 4, 2, padding="same")(b)
    d4 = layers.BatchNormalization()(d4); d4 = layers.LeakyReLU(0.2)(d4)
    d4 = crop_to_match(d4, e4); d4 = layers.Concatenate()([d4, e4])

    d3 = layers.Conv1DTranspose(256, 4, 2, padding="same")(d4)
    d3 = layers.BatchNormalization()(d3); d3 = layers.LeakyReLU(0.2)(d3)
    d3 = crop_to_match(d3, e3); d3 = layers.Concatenate()([d3, e3])

    d2 = layers.Conv1DTranspose(128, 4, 2, padding="same")(d3)
    d2 = layers.BatchNormalization()(d2); d2 = layers.LeakyReLU(0.2)(d2)
    d2 = crop_to_match(d2, e2); d2 = layers.Concatenate()([d2, e2])

    d1 = layers.Conv1DTranspose(64, 4, 2, padding="same")(d2)
    d1 = layers.BatchNormalization()(d1); d1 = layers.LeakyReLU(0.2)(d1)
    d1 = crop_to_match(d1, e1); d1 = layers.Concatenate()([d1, e1])

    # ===== Output =====
    out = layers.Conv1DTranspose(32, 4, 2, padding="same")(d1)
    out = layers.LeakyReLU(0.2)(out); out = crop_to_match(out, x)


    out = layers.Conv1D(1, 3, padding="same", activation="sigmoid")(out)
    out = layers.Flatten()(out)


    out = layers.Lambda(lambda t: tf.ensure_shape(t, [None, out_dim]), name="clean_spectrum")(out)

    return Model(inp, out, name="Generator")
# ---------------------------------

# ---------- Discriminator ----------
def build_discriminator(spec_dim, ref_dim):
    spec_inp = Input(shape=(spec_dim,))
    ref_inp = Input(shape=(ref_dim,))

    sx = layers.Reshape((spec_dim, 1))(spec_inp)
    rx = layers.Reshape((ref_dim, 1))(ref_inp)
    x  = layers.Concatenate(axis=1)([sx, rx])


    for i, f in enumerate([64, 128, 128, 256]):
        x = layers.Conv1D(f, kernel_size=5, strides=2, padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.LeakyReLU(0.2)(x)


    feat  = layers.GlobalAveragePooling1D()(x)
    dense = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001))(feat)
    dense = layers.Dropout(0.3)(dense)
    out   = layers.Dense(1, activation="sigmoid")(dense)

    D  = Model([spec_inp, ref_inp], out,  name="Discriminator")
    Fe = Model([spec_inp, ref_inp], feat, name="FeatureExtractor")
    return D, Fe

# ----------------------------------

# ---------- Conditional GAN ----------
class ConditionalGAN(Model):
    def __init__(self, spec_dim, ref_dim, Zn_idx, Ni_idx, Cu_idx, lr=1e-4):
        super().__init__()
        self.generator, self.discriminator, self.feature_extractor = (
            build_generator(ref_dim, spec_dim),
            *build_discriminator(spec_dim, ref_dim)
        )
        self.g_opt = Adam(0.5*lr)
        self.d_opt = Adam(lr)
        self.bce   = BinaryCrossentropy(from_logits=False)

        self.Zn, self.Ni, self.Cu = Zn_idx, Ni_idx, Cu_idx
        self.l_adv_log   = tf.Variable(tf.math.log(1.0),  True)
        self.l_cos_log   = tf.Variable(tf.math.log(1.0),  True)
        self.l_feat_log  = tf.Variable(tf.math.log(30.0), True)
        self.l_metal_log = tf.Variable(tf.math.log(50.0), True)


        self.g_optimizer = self.g_opt
        self.d_optimizer = self.d_opt


    def train_step(self, data):
        clean, x = data
        # ---- D ----
        with tf.GradientTape() as td:
            fake = self.generator(x, training=True)
            r, f = self.discriminator([clean, x], True), self.discriminator([fake, x], True)
            d_loss = self.bce(tf.ones_like(r)*0.9, r) + self.bce(tf.zeros_like(f), f)
        self.d_opt.apply_gradients(zip(td.gradient(d_loss, self.discriminator.trainable_variables),
                                       self.discriminator.trainable_variables))
        # ---- G (+λ) ----
        with tf.GradientTape() as tg:
            fake   = self.generator(x, training=True)
            f_pred = self.discriminator([fake, x], True)

            adv  = self.bce(tf.ones_like(f_pred), f_pred)
            cos  = tf.reduce_mean(1+tf.keras.losses.cosine_similarity(fake, clean, axis=-1))
            feat = tf.reduce_mean(tf.abs(
                   self.feature_extractor([clean,x],False) -
                   self.feature_extractor([fake,x],False)))

            real_m = tf.concat([clean[:,self.Zn[0]:self.Zn[1]],
                                clean[:,self.Ni[0]:self.Ni[1]],
                                clean[:,self.Cu[0]:self.Cu[1]]],1)
            fake_m = tf.concat([fake [:,self.Zn[0]:self.Zn[1]],
                                fake [:,self.Ni[0]:self.Ni[1]],
                                fake [:,self.Cu[0]:self.Cu[1]]],1)
            metal = tf.reduce_mean(tf.abs(fake_m - real_m))

            l_adv   = tf.nn.softplus(self.l_adv_log)
            l_cos   = tf.nn.softplus(self.l_cos_log)
            l_feat  = tf.nn.softplus(self.l_feat_log)
            l_metal = tf.nn.softplus(self.l_metal_log)

            g_loss = l_adv*adv + l_cos*cos + l_feat*feat + l_metal*metal

        vars_g = (self.generator.trainable_variables +
                  [self.l_adv_log, self.l_cos_log, self.l_feat_log, self.l_metal_log])
        self.g_opt.apply_gradients(zip(tg.gradient(g_loss, vars_g), vars_g))

        D_real_mean = tf.reduce_mean(r)
        D_fake_mean = tf.reduce_mean(f)

        return {
            "d_loss": d_loss, "g_loss": g_loss,
            "adv": adv, "cos": cos, "feat": feat, "metal": metal,
            "λ_adv": l_adv, "λ_cos": l_cos, "λ_feat": l_feat, "λ_metal": l_metal,
            "D_real": D_real_mean,
            "D_fake": D_fake_mean
        }


## Restore checkpoint state and continue training

Original code cell 7.


In [ ]:
# === Import & Setup ===
import os, numpy as np, tensorflow as tf

additional_epochs = 100
learning_rate = 1e-4


cgan_transfer = ConditionalGAN(
    spec_dim = int(training_target.shape[1]),
    ref_dim  = int(training_spectra.shape[1]),
    Zn_idx   = Zn_index,
    Ni_idx   = Ni_index,
    Cu_idx   = Cu_index,
    lr       = learning_rate
)


ckpt_kwargs = {
    "generator": cgan_transfer.generator,
    "discriminator": cgan_transfer.discriminator,
    "g_opt": getattr(cgan_transfer, "g_optimizer", getattr(cgan_transfer, "g_opt", None)),
    "d_opt": getattr(cgan_transfer, "d_optimizer", getattr(cgan_transfer, "d_opt", None)),
    "l_adv_log": cgan_transfer.l_adv_log,
    "l_cos_log": cgan_transfer.l_cos_log,
    "l_feat_log": cgan_transfer.l_feat_log,
    "l_metal_log": cgan_transfer.l_metal_log
}
ckpt_kwargs = {k: v for k, v in ckpt_kwargs.items() if v is not None}
ckpt2 = tf.train.Checkpoint(**ckpt_kwargs)


first_ckpt = str(DATA_DIR / "checkpoints/ckpt/ckpt-1")
ckpt2.restore(first_ckpt).expect_partial()
print(f"[INFO] Restored from fixed checkpoint: {first_ckpt}")


prev_hist = {}
if (DATA_DIR / "checkpoints/history.npz").exists():
    try:
        prev_hist = dict(np.load(DATA_DIR / "checkpoints/history.npz", allow_pickle=True))
        print("[INFO] Loaded previous training history.")
    except:
        pass


cont_history = {k: [] for k in [
    "d_loss", "g_loss", "adv", "cos", "feat", "metal",
    "λ_adv", "λ_cos", "λ_feat", "λ_metal", "D_real", "D_fake"
]}


for epoch in range(additional_epochs):
    epoch_losses = {k: [] for k in cont_history}

    idx = tf.random.shuffle(tf.range(training_spectra.shape[0]))
    inter_shuf = tf.gather(training_spectra, idx)
    clean_shuf = tf.gather(training_target, idx)

    for i in range(0, training_spectra.shape[0], batch_size):
        real_batch = clean_shuf[i:i+batch_size]
        inter_batch = inter_shuf[i:i+batch_size]
        losses = cgan_transfer.train_step((real_batch, inter_batch))

        for k in cont_history:
            v = losses.get(k, None)
            if v is None: continue
            try:
                if hasattr(v, "numpy"):
                    v = v.numpy()
                v = float(np.mean(v)) if np.ndim(v) > 0 else float(v)
            except: continue
            epoch_losses[k].append(v)

    mean_losses = {k: (float(np.mean(v)) if len(v) else float("nan"))
                   for k, v in epoch_losses.items()}

    def fmt(key, fmtstr="{:.4e}"):
        val = mean_losses.get(key, float("nan"))
        return f"{key}: {('nan' if np.isnan(val) else fmtstr.format(val))}"

    log_str = f"[CONT] Epoch {epoch+1:>3}/{additional_epochs} | " + " | ".join([
        fmt("d_loss"), fmt("g_loss"), fmt("adv"), fmt("cos"), fmt("feat"), fmt("metal")
    ])
    log_str += f" | D_real: {mean_losses.get('D_real', float('nan')):.3f} | D_fake: {mean_losses.get('D_fake', float('nan')):.3f}"
    log_str += f" | λ_adv: {mean_losses.get('λ_adv', float('nan')):.2f} " \
               f"λ_cos: {mean_losses.get('λ_cos', float('nan')):.2f} " \
               f"λ_feat: {mean_losses.get('λ_feat', float('nan')):.2f} " \
               f"λ_metal: {mean_losses.get('λ_metal', float('nan')):.2f}"
    print(log_str)

    for k in cont_history:
        cont_history[k].append(mean_losses.get(k, float("nan")))


cgan_transfer.generator.save_weights("checkpoints/generator_weights_continued.h5")
cgan_transfer.discriminator.save_weights("checkpoints/discriminator_weights_continued.h5")


save_path = os.path.join("checkpoints/ckpt", "ckpt-2")
ckpt2.write(save_path)
print(f"[INFO] Manually saved continued checkpoint to: {save_path}")


merged_history = {}
all_keys = set(list(prev_hist.keys()) + list(cont_history.keys()))
for k in all_keys:
    merged_history[k] = list(prev_hist.get(k, [])) + list(cont_history.get(k, []))

np.savez("checkpoints/history_continued.npz", **cont_history)
np.savez("checkpoints/history_all.npz", **merged_history)
print("[INFO] Saved history_continued.npz and history_all.npz")


## Inspect training history

Original code cell 8.


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(merged_history["D_real"], label="D Real Pred")
plt.plot(merged_history["D_fake"], label="D Fake Pred")
plt.xlabel("Epoch")
plt.ylabel("D Output (Mean)")
plt.legend()
plt.grid(True)
plt.title("Discriminator Output over Epochs")
plt.show()


plt.figure(figsize=(8, 5))
plt.plot(merged_history["λ_adv"],   label="λ_adv")
plt.plot(merged_history["λ_cos"],   label="λ_cos")
plt.plot(merged_history["λ_feat"],  label="λ_feat")
plt.plot(merged_history["λ_metal"], label="λ_metal")
plt.xlabel("Epoch")
plt.ylabel("Lambda Value")
plt.legend()
plt.grid(True)
plt.title("Learnable Lambda Weights over Epochs")
plt.show()

# === Generator Losses ===
plt.figure(figsize=(12, 5))
plt.plot(merged_history['g_loss'], label='Total Generator Loss')
plt.plot(merged_history['adv'], label='Adversarial Loss')
plt.plot(merged_history['cos'], label='Cosine Loss')
plt.plot(merged_history['feat'], label='Feature Loss')
plt.plot(merged_history['metal'], label='Metal Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Generator Loss Components')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# === Discriminator Loss ===
plt.figure(figsize=(6, 5))
plt.plot(merged_history['d_loss'], label='Discriminator Loss', color='tab:red')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Discriminator Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## Inspect a training sample

Original code cell 9.


In [ ]:
meta_cols = X_cols[:4]
idx_map   = {name: i for i, name in enumerate(meta_cols)}
print("🗂  Meta column order:", meta_cols)
# =========================================================


target = dict(Cr=1, Zn=5, Ni=5, Cu=5)
# ---------------------------------------------------------


train_mask = np.ones(len(training_condition), dtype=bool)
for metal, ppm in target.items():
    train_mask &= (training_condition[:, idx_map[metal]] == ppm)

train_indices = tf.where(train_mask)[:, 0]
train_idx = train_indices[0].numpy() if len(train_indices) else None
# ---------------------------------------------------------

def plot_with_zoom_metal_regions(idx,
                                 clean_spec, interfered_spec,
                                 cond, dataset_name):

    real_spec = clean_spec[idx].numpy()
    fake_spec = cgan_transfer.generator(interfered_spec[idx:idx+1],
                               training=False).numpy().squeeze()
    label = cond[idx]

    title = (f"{dataset_name} - Real vs. Fake Spectrum\n" +
             " ".join([f"{m}={label[idx_map[m]]:.0f}ppm"
                       for m in ['Cr','Cu','Zn','Ni']]))


    fig, axs = plt.subplots(4, 1, figsize=(12, 10),
                            gridspec_kw={'height_ratios': [2,1,1,1]})
    fig.suptitle(title, fontsize=14)


    axs[0].plot(wavelength, real_spec, label='Real',  c='k')
    axs[0].plot(wavelength, fake_spec, label='Fake',  c='r', ls='--')
    axs[0].legend(); axs[0].grid(True)

    for rng in [Zn_index, Ni_index, Cu_index]:
        axs[0].axvspan(wavelength[rng[0]], wavelength[rng[1]-1],
                       color='grey', alpha=.2)


    axs[1].plot(wavelength[Zn_index[0]:Zn_index[1]],
                real_spec[Zn_index[0]:Zn_index[1]], c='k')
    axs[1].plot(wavelength[Zn_index[0]:Zn_index[1]],
                fake_spec[Zn_index[0]:Zn_index[1]], c='r', ls='--')
    axs[1].set_ylabel('Zn');  axs[1].grid(True)

    axs[2].plot(wavelength[Ni_index[0]:Ni_index[1]],
                real_spec[Ni_index[0]:Ni_index[1]], c='k')
    axs[2].plot(wavelength[Ni_index[0]:Ni_index[1]],
                fake_spec[Ni_index[0]:Ni_index[1]], c='r', ls='--')
    axs[2].set_ylabel('Ni');  axs[2].grid(True)

    axs[3].plot(wavelength[Cu_index[0]:Cu_index[1]],
                real_spec[Cu_index[0]:Cu_index[1]], c='k')
    axs[3].plot(wavelength[Cu_index[0]:Cu_index[1]],
                fake_spec[Cu_index[0]:Cu_index[1]], c='r', ls='--')
    axs[3].set_ylabel('Cu');  axs[3].set_xlabel('Wavelength (nm)')
    axs[3].grid(True)

    plt.tight_layout(rect=[0,0,1,0.96])
    plt.show()


if train_idx is None:
    print("No matching training sample. Check the target values.")
else:
    print("Selected training index:", train_idx)
    plot_with_zoom_metal_regions(train_idx,
                                 training_target,
                                 training_spectra,
                                 training_condition,
                                 "Training")


## Inspect a test sample

Original code cell 10.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches


target_Cr = 4      # Cr ppm
target_Cu = 2      # Cu ppm
target_Zn = 2      # Zn ppm
target_Ni = 2      # Ni ppm


test_mask = (
    (testing_condition[:, 0] == target_Cr) &   # Cr
    (testing_condition[:, 1] == target_Cu) &   # Cu
    (testing_condition[:, 2] == target_Zn) &   # Zn
    (testing_condition[:, 3] == target_Ni)     # Ni
)
test_idx_arr = tf.where(test_mask)[:, 0]
test_idx = test_idx_arr[0].numpy() if len(test_idx_arr) > 0 else None

def to_np(t):
    return t.numpy() if hasattr(t, "numpy") else t

def plot_with_zoom(idx, clean_spec, interfered_spec, meta_info, set_name):
    """Plot the full spectrum and enlarged Zn, Ni, and Cu regions."""
    real  = to_np(clean_spec[idx:idx+1]).squeeze()
    inp   = interfered_spec[idx:idx+1]
    label = meta_info[idx:idx+1]
    fake  = cgan_transfer.generator(inp, training=False).numpy().squeeze()

    title = (f"{set_name} - Real vs Fake\n"
             f"(Cr={label[0,0]:.0f}, Cu={label[0,1]:.0f}, Zn={label[0,2]:.0f}, Ni={label[0,3]:.0f})")

    fig, ax = plt.subplots(4, 1, figsize=(12, 10),
                           gridspec_kw={'height_ratios': [2, 1, 1, 1]}, sharex=False)
    fig.suptitle(title, fontsize=14)


    ax[0].plot(wavelength, real, label='Real', color='black', lw=1)
    ax[0].plot(wavelength, fake, label='Fake', color='red', ls='--', lw=1)
    ax[0].set_ylabel("Intensity"); ax[0].legend(); ax[0].grid(True)
    for band in [Zn_index, Ni_index, Cu_index]:
        ax[0].axvspan(wavelength[band[0]], wavelength[band[1]-1],
                      color='gray', alpha=0.2)

    # --- (2) Zn zoom ---
    ax[1].plot(wavelength[Zn_index[0]:Zn_index[1]],
               real[Zn_index[0]:Zn_index[1]], color='black')
    ax[1].plot(wavelength[Zn_index[0]:Zn_index[1]],
               fake[Zn_index[0]:Zn_index[1]], color='red', ls='--')
    ax[1].set_ylabel("Zn"); ax[1].grid(True)

    # --- (3) Ni zoom ---
    ax[2].plot(wavelength[Ni_index[0]:Ni_index[1]],
               real[Ni_index[0]:Ni_index[1]], color='black')
    ax[2].plot(wavelength[Ni_index[0]:Ni_index[1]],
               fake[Ni_index[0]:Ni_index[1]], color='red', ls='--')
    ax[2].set_ylabel("Ni"); ax[2].grid(True)

    # --- (4) Cu zoom ---
    ax[3].plot(wavelength[Cu_index[0]:Cu_index[1]],
               real[Cu_index[0]:Cu_index[1]], color='black')
    ax[3].plot(wavelength[Cu_index[0]:Cu_index[1]],
               fake[Cu_index[0]:Cu_index[1]], color='red', ls='--')
    ax[3].set_ylabel("Cu"); ax[3].set_xlabel("Wavelength (nm)"); ax[3].grid(True)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()


if test_idx is None:
    print("No matching test sample. Check the Cr, Cu, Zn, and Ni target settings.")
else:
    print(f"Drawing testing spectrum at index: {test_idx}")
    plot_with_zoom(test_idx, testing_target, testing_spectra, testing_condition, "Testing")


## Export restored spectral tables

Original code cell 11.


In [ ]:
# ==============================  EXPORT FAKE SPECTRA  ==============================


meta_cols = X_cols[:4]

# ------------ Training set ------------
train_rows = []
for idx in range(training_spectra.shape[0]):
    fake_full = cgan_transfer.generator(training_spectra[idx: idx+1], training=False).numpy().squeeze()
    meta_vals = training_condition[idx]
    train_rows.append(np.concatenate([meta_vals, fake_full]))

train_df = pd.DataFrame(
    train_rows,
    columns=list(meta_cols) + [f'{wl:.3f}' for wl in wavelength]
)
train_df.to_csv("training_fake_spectrum_full(Transfer).csv", index=False)
print("✅ training_fake_spectrum_full(Transfer).csv  saved (columns aligned)")

# ------------ Testing set --------------
test_rows = []
for idx in range(testing_spectra.shape[0]):
    fake_full = cgan_transfer.generator(testing_spectra[idx: idx+1], training=False).numpy().squeeze()
    meta_vals = testing_condition[idx]
    test_rows.append(np.concatenate([meta_vals, fake_full]))

test_df = pd.DataFrame(
    test_rows,
    columns=list(meta_cols) + [f'{wl:.3f}' for wl in wavelength]
)
test_df.to_csv("testing_fake_spectrum_full(Transfer).csv", index=False)
print("✅ testing_fake_spectrum_full(Transfer).csv   saved (columns aligned)")

# ------------ Retake --------------
retake_rows = []
for idx in range(retake_spectra.shape[0]):
    fake_full = cgan_transfer.generator(retake_spectra[idx: idx+1], training=False).numpy().squeeze()
    meta_vals = retake_condition[idx]
    retake_rows.append(np.concatenate([meta_vals, fake_full]))

retake_df = pd.DataFrame(
    retake_rows,
    columns=list(meta_cols) + [f'{wl:.3f}' for wl in wavelength]
)
retake_df.to_csv("retake_fake_spectrum_full(Transfer).csv", index=False)
print("✅ retake_fake_spectrum_full(Transfer).csv   saved (columns aligned)")

# ------------ Wastewater --------------
wastewater_rows = []
for idx in range(wastewater_spectra.shape[0]):
    fake_full = cgan_transfer.generator(wastewater_spectra[idx: idx+1], training=False).numpy().squeeze()
    meta_vals = wastewater_condition[idx]
    wastewater_rows.append(np.concatenate([meta_vals, fake_full]))

wastewater_df = pd.DataFrame(
    wastewater_rows,
    columns=list(meta_cols) + [f'{wl:.3f}' for wl in wavelength]
)
wastewater_df.to_csv("wastewater_fake_spectrum_full(Transfer).csv", index=False)
print("✅ wastewater_fake_spectrum_full(Transfer).csv   saved (columns aligned)")


## Export repeat-measurement figures

Original code cell 12.


In [ ]:
# ============================  SAVE ALL Retake FIGURES  ============================
import os, tqdm, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

# --------------------------------------------------------------------------

print("🔄  Generating fake spectra ...")
fake_test = cgan_transfer.generator.predict(
    retake_spectra, batch_size=512, verbose=0
)                                   # shape = (N, spec_dim)
print("   Done. Shape:", fake_test.shape)

# --------------------------------------------------------------------------

print("🔄  Computing global y-axis ranges ...")


all_real_spectra = retake_target.numpy() if hasattr(retake_target, 'numpy') else retake_target
all_fake_spectra = fake_test


def compute_global_ranges():

    global_min = min(np.min(all_real_spectra), np.min(all_fake_spectra))
    global_max = max(np.max(all_real_spectra), np.max(all_fake_spectra))
    full_range = (global_min * 0.95, global_max * 1.05)


    zn_real = all_real_spectra[:, Zn_index[0]:Zn_index[1]]
    zn_fake = all_fake_spectra[:, Zn_index[0]:Zn_index[1]]
    zn_min = min(np.min(zn_real), np.min(zn_fake))
    zn_max = max(np.max(zn_real), np.max(zn_fake))
    zn_range = (zn_min * 0.95, zn_max * 1.05)


    ni_real = all_real_spectra[:, Ni_index[0]:Ni_index[1]]
    ni_fake = all_fake_spectra[:, Ni_index[0]:Ni_index[1]]
    ni_min = min(np.min(ni_real), np.min(ni_fake))
    ni_max = max(np.max(ni_real), np.max(ni_fake))
    ni_range = (ni_min * 0.95, ni_max * 1.05)


    cu_real = all_real_spectra[:, Cu_index[0]:Cu_index[1]]
    cu_fake = all_fake_spectra[:, Cu_index[0]:Cu_index[1]]
    cu_min = min(np.min(cu_real), np.min(cu_fake))
    cu_max = max(np.max(cu_real), np.max(cu_fake))
    cu_range = (cu_min * 0.95, cu_max * 1.05)

    return full_range, zn_range, ni_range, cu_range


full_ylim, zn_ylim, ni_ylim, cu_ylim = compute_global_ranges()

print(f"   Full spectrum range: {full_ylim[0]:.3f} to {full_ylim[1]:.3f}")
print(f"   Zn region range:     {zn_ylim[0]:.3f} to {zn_ylim[1]:.3f}")
print(f"   Ni region range:     {ni_ylim[0]:.3f} to {ni_ylim[1]:.3f}")
print(f"   Cu region range:     {cu_ylim[0]:.3f} to {cu_ylim[1]:.3f}")

# --------------------------------------------------------------------------

def plot_with_zoom_fixed_axes(idx, real, fake, meta, save_path):
    fig, axs = plt.subplots(4,1,figsize=(12,10),
                            gridspec_kw={'height_ratios':[2,1,1,1]})
    title = ("retake – Real vs Fake\n" +
             " ".join([f"{m}={meta[idx_map[m]]:.0f}ppm"
                       for m in desired_order]))
    fig.suptitle(title, fontsize=14)


    axs[0].plot(wavelength, real, c='k', label='Real')
    axs[0].plot(wavelength, fake, c='r', ls='--', label='Fake')
    axs[0].set_ylim(full_ylim)
    axs[0].legend(); axs[0].grid(True)
    for rng in [Zn_index, Ni_index, Cu_index]:
        axs[0].axvspan(wavelength[rng[0]], wavelength[rng[1]-1],
                       color='grey', alpha=.2)


    axs[1].plot(wavelength[Zn_index[0]:Zn_index[1]],
                real[Zn_index[0]:Zn_index[1]], c='k')
    axs[1].plot(wavelength[Zn_index[0]:Zn_index[1]],
                fake[Zn_index[0]:Zn_index[1]], c='r', ls='--')
    axs[1].set_ylim(zn_ylim)
    axs[1].set_ylabel('Zn'); axs[1].grid(True)


    axs[2].plot(wavelength[Ni_index[0]:Ni_index[1]],
                real[Ni_index[0]:Ni_index[1]], c='k')
    axs[2].plot(wavelength[Ni_index[0]:Ni_index[1]],
                fake[Ni_index[0]:Ni_index[1]], c='r', ls='--')
    axs[2].set_ylim(ni_ylim)
    axs[2].set_ylabel('Ni'); axs[2].grid(True)


    axs[3].plot(wavelength[Cu_index[0]:Cu_index[1]],
                real[Cu_index[0]:Cu_index[1]], c='k')
    axs[3].plot(wavelength[Cu_index[0]:Cu_index[1]],
                fake[Cu_index[0]:Cu_index[1]], c='r', ls='--')
    axs[3].set_ylim(cu_ylim)
    axs[3].set_ylabel('Cu'); axs[3].set_xlabel('Wavelength (nm)')
    axs[3].grid(True)

    plt.tight_layout(rect=[0,0,1,0.96])
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

# --------------------------------------------------------------------------

desired_order = ['Cr','Zn','Ni','Cu']
idx_map = {m: int(np.where(np.array(X_cols[:4]) == m)[0][0])
           for m in desired_order}
out_dir = "retake prediction plots"
os.makedirs(out_dir, exist_ok=True)

# --------------------------------------------------------------------------

print("🔄  Saving PNG files with fixed y-axis ranges ...")
for idx in tqdm.tqdm(range(retake_spectra.shape[0]), desc="saving png"):
    save_path = os.path.join(out_dir, f"test_{idx:05d}.png")
    plot_with_zoom_fixed_axes(idx,
                              retake_target[idx].numpy() if hasattr(retake_target[idx], 'numpy') else retake_target[idx],
                              fake_test[idx],
                              retake_condition[idx],
                              save_path)

print("Saved all PNGs to the retake prediction plots directory.")
print("All plots use consistent vertical axis limits.")


## Export wastewater figures

Original code cell 13.


In [ ]:
# ============================  SAVE ALL Wastewater FIGURES  ============================
import os, tqdm, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

# --------------------------------------------------------------------------

print("🔄  Generating fake spectra ...")
fake_test = cgan_transfer.generator.predict(
    wastewater_spectra, batch_size=512, verbose=0
)                                   # shape = (N, spec_dim)
print("   Done. Shape:", fake_test.shape)

# --------------------------------------------------------------------------

print("🔄  Computing global y-axis ranges ...")


all_real_spectra = wastewater_target.numpy() if hasattr(wastewater_target, 'numpy') else wastewater_target
all_fake_spectra = fake_test


def compute_global_ranges():

    global_min = min(np.min(all_real_spectra), np.min(all_fake_spectra))
    global_max = max(np.max(all_real_spectra), np.max(all_fake_spectra))
    full_range = (global_min * 0.95, global_max * 1.05)


    zn_real = all_real_spectra[:, Zn_index[0]:Zn_index[1]]
    zn_fake = all_fake_spectra[:, Zn_index[0]:Zn_index[1]]
    zn_min = min(np.min(zn_real), np.min(zn_fake))
    zn_max = max(np.max(zn_real), np.max(zn_fake))
    zn_range = (zn_min * 0.95, zn_max * 1.05)


    ni_real = all_real_spectra[:, Ni_index[0]:Ni_index[1]]
    ni_fake = all_fake_spectra[:, Ni_index[0]:Ni_index[1]]
    ni_min = min(np.min(ni_real), np.min(ni_fake))
    ni_max = max(np.max(ni_real), np.max(ni_fake))
    ni_range = (ni_min * 0.95, ni_max * 1.05)


    cu_real = all_real_spectra[:, Cu_index[0]:Cu_index[1]]
    cu_fake = all_fake_spectra[:, Cu_index[0]:Cu_index[1]]
    cu_min = min(np.min(cu_real), np.min(cu_fake))
    cu_max = max(np.max(cu_real), np.max(cu_fake))
    cu_range = (cu_min * 0.95, cu_max * 1.05)

    return full_range, zn_range, ni_range, cu_range


full_ylim, zn_ylim, ni_ylim, cu_ylim = compute_global_ranges()

print(f"   Full spectrum range: {full_ylim[0]:.3f} to {full_ylim[1]:.3f}")
print(f"   Zn region range:     {zn_ylim[0]:.3f} to {zn_ylim[1]:.3f}")
print(f"   Ni region range:     {ni_ylim[0]:.3f} to {ni_ylim[1]:.3f}")
print(f"   Cu region range:     {cu_ylim[0]:.3f} to {cu_ylim[1]:.3f}")

# --------------------------------------------------------------------------

def plot_with_zoom_fixed_axes(idx, real, fake, meta, save_path):
    fig, axs = plt.subplots(4,1,figsize=(12,10),
                            gridspec_kw={'height_ratios':[2,1,1,1]})
    title = ("wastewater – Real vs Fake\n" +
             " ".join([f"{m}={meta[idx_map[m]]:.0f}ppm"
                       for m in desired_order]))
    fig.suptitle(title, fontsize=14)


    axs[0].plot(wavelength, real, c='k', label='Real')
    axs[0].plot(wavelength, fake, c='r', ls='--', label='Fake')
    axs[0].set_ylim(full_ylim)
    axs[0].legend(); axs[0].grid(True)
    for rng in [Zn_index, Ni_index, Cu_index]:
        axs[0].axvspan(wavelength[rng[0]], wavelength[rng[1]-1],
                       color='grey', alpha=.2)


    axs[1].plot(wavelength[Zn_index[0]:Zn_index[1]],
                real[Zn_index[0]:Zn_index[1]], c='k')
    axs[1].plot(wavelength[Zn_index[0]:Zn_index[1]],
                fake[Zn_index[0]:Zn_index[1]], c='r', ls='--')
    axs[1].set_ylim(zn_ylim)
    axs[1].set_ylabel('Zn'); axs[1].grid(True)


    axs[2].plot(wavelength[Ni_index[0]:Ni_index[1]],
                real[Ni_index[0]:Ni_index[1]], c='k')
    axs[2].plot(wavelength[Ni_index[0]:Ni_index[1]],
                fake[Ni_index[0]:Ni_index[1]], c='r', ls='--')
    axs[2].set_ylim(ni_ylim)
    axs[2].set_ylabel('Ni'); axs[2].grid(True)


    axs[3].plot(wavelength[Cu_index[0]:Cu_index[1]],
                real[Cu_index[0]:Cu_index[1]], c='k')
    axs[3].plot(wavelength[Cu_index[0]:Cu_index[1]],
                fake[Cu_index[0]:Cu_index[1]], c='r', ls='--')
    axs[3].set_ylim(cu_ylim)
    axs[3].set_ylabel('Cu'); axs[3].set_xlabel('Wavelength (nm)')
    axs[3].grid(True)

    plt.tight_layout(rect=[0,0,1,0.96])
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

# --------------------------------------------------------------------------

desired_order = ['Cr','Zn','Ni','Cu']
idx_map = {m: int(np.where(np.array(X_cols[:4]) == m)[0][0])
           for m in desired_order}
out_dir = "wastewater prediction plots"
os.makedirs(out_dir, exist_ok=True)

# --------------------------------------------------------------------------

print("🔄  Saving PNG files with fixed y-axis ranges ...")
for idx in tqdm.tqdm(range(wastewater_spectra.shape[0]), desc="saving png"):
    save_path = os.path.join(out_dir, f"test_{idx:05d}.png")
    plot_with_zoom_fixed_axes(idx,
                              wastewater_target[idx].numpy() if hasattr(wastewater_target[idx], 'numpy') else wastewater_target[idx],
                              fake_test[idx],
                              wastewater_condition[idx],
                              save_path)

print("Saved all PNGs to the wastewater prediction plots directory.")
print("All plots use consistent vertical axis limits.")
